# 03 — Modeling

**Goal**: Train all models and build a benchmark comparison table.

## Models

| Model | Features | Target | Task |
|-------|----------|--------|------|
| Null (mean) | none | CAR[0,3] | regression baseline |
| Market-only | pre_vol | CAR[0,3] | regression |
| LM Lexicon OLS | NegRate + PosRate | CAR[0,3] | regression |
| LM + Controls | + pre_vol + year FE | CAR[0,3] | regression |
| FinBERT OLS | finbert_neg + finbert_pos | CAR[0,3] | regression |
| Naive Bayes | TF-IDF (500 feats) | direction | classification |
| Logistic Regression | TF-IDF (500 feats) | direction | classification |

**Train/Test split**: 2016–2018 train | 2019–2020 test (time-based, no look-ahead).

In [ ]:
import os, sys
from pathlib import Path

# Navigate to project root so relative paths work correctly
# When running from notebooks/, we need to go up one level
project_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd()
# Fallback: search upward for pyproject.toml
for p in [project_root] + list(project_root.parents):
    if (p / "pyproject.toml").exists():
        project_root = p
        break
os.chdir(project_root)
# Add src/ to Python path so nasdaq_nlp imports work even without install
sys.path.insert(0, str(project_root / "src"))
print(f"Working directory: {Path.cwd()}")

## Regression Models (OLS)

**Model specification:**

$$\text{CAR}_{i,[0,3]} = \beta_0 + \beta_1 \cdot \text{NegRate}_i + \beta_2 \cdot \text{PosRate}_i + \text{controls} + \varepsilon_i$$

**Evaluation metric: R²**

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- In-sample R²: fit on training data (2016-2018)
- Out-of-sample R²: predictions on test data (2019-2020), using training mean as the benchmark

In [ ]:
from nasdaq_nlp.models.regression import run_regression_pipeline

benchmark_df, regression_results = run_regression_pipeline()
print("\n=== Benchmark Table ===")
print(benchmark_df.to_string(index=False))

## Classification Models (Sessions 9-10)

For classification, we convert the regression problem into a binary prediction:

$$y_i = \begin{cases} 1 & \text{if } \text{CAR}_{i,[0,3]} > 0 \quad \text{(stock outperformed)} \\ 0 & \text{otherwise} \end{cases}$$

**Naive Bayes** (Session 9):
$$P(y \mid x) \propto P(y) \prod_i P(x_i \mid y)$$

Assumes features are conditionally independent given the class label (the "naive" assumption).

**Logistic Regression** (Session 10):
$$P(y=1 \mid x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \ldots + \beta_n x_n)}}$$

Models the probability of a positive market reaction directly.

In [ ]:
from nasdaq_nlp.models.classifiers import run_classifiers

nb_results, lr_results = run_classifiers()

# Display as a table
import pandas as pd
clf_table = pd.DataFrame([nb_results, lr_results])
print("\n=== Classifier Results ===")
print(clf_table[['model','train_accuracy','test_accuracy','test_f1','test_precision','test_recall']].to_string(index=False))

In [ ]:
# Combined benchmark view (regression + classification)
import pandas as pd

reg_rows = []
for r in regression_results:
    if r.target == 'car_03':
        reg_rows.append({
            'Model': r.model_name,
            'Task': 'regression',
            'Target': 'CAR[0,3]',
            'Train metric': f"R²={r.train_r2:.3f}",
            'Test metric': f"OOS R²={r.oos_r2:.3f}",
        })

clf_rows = [
    {'Model': nb_results['model'], 'Task': 'classification', 'Target': 'direction',
     'Train metric': f"acc={nb_results['train_accuracy']:.3f}",
     'Test metric': f"acc={nb_results['test_accuracy']:.3f}"},
    {'Model': lr_results['model'], 'Task': 'classification', 'Target': 'direction',
     'Train metric': f"acc={lr_results['train_accuracy']:.3f}",
     'Test metric': f"acc={lr_results['test_accuracy']:.3f}"},
]

combined = pd.DataFrame(reg_rows + clf_rows)
print(combined.to_string(index=False))

## Sanity Checks

In [ ]:
# All regression results should have valid R² values
for r in regression_results:
    assert r.train_r2 is not None, f"{r.model_name} missing train R²"
    assert r.oos_r2 is not None, f"{r.model_name} missing OOS R²"

# Classifiers should beat the 50% random baseline
assert nb_results['test_accuracy'] > 0.50, "NB below random baseline"
assert lr_results['test_accuracy'] > 0.50, "LR below random baseline"

print("✓ All models trained and evaluated successfully")
print(f"  Regression models: {len(regression_results)}")
print(f"  Classification models: 2 (NB + LR)")